In [ ]:
from pathlib import Path
import re
import html
import unicodedata
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "Models").exists() and (candidate / "Dataset source").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing Models and Dataset source.")

PROJECT_ROOT = find_project_root(Path.cwd())
SOURCE_DIR = PROJECT_ROOT / "Dataset source"
BASE_DIR = SOURCE_DIR / "training"
BASE_DIR.mkdir(parents=True, exist_ok=True)
file_path = SOURCE_DIR / "SPAM text message 20170820 - Data.csv"
df = pd.read_csv(file_path, encoding="latin-1", usecols=[0, 1])
df.columns = ["Category", "Message"]


In [ ]:
def clean_message(text):
    s = unicodedata.normalize("NFKC", html.unescape(str(text))).strip()

    # Protect useful patterns before removing punctuation and special characters.
    s = re.sub(r"<\s*(?:#|num)\s*>", " __NUM_TOKEN__ ", s, flags=re.I)
    s = re.sub(r"<\s*url\s*>", " __URL_TOKEN__ ", s, flags=re.I)
    s = re.sub(r"<\s*cur\s*>", " __CUR_TOKEN__ ", s, flags=re.I)
    s = re.sub(r"\b(?:https?://|www\.)\S+", " __URL_TOKEN__ ", s, flags=re.I)
    s = re.sub(r"0A\$", " __CUR_TOKEN__ ", s, flags=re.I)
    s = re.sub(r"[$£€¥₹]", " __CUR_TOKEN__ ", s)

    # Apply lowercase conversion and noise removal following common text-cleaning practice.
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)

    # Restore consistent placeholders after noise removal.
    s = s.replace("__num_token__", "<NUM>")
    s = s.replace("__url_token__", "<URL>")
    s = s.replace("__cur_token__", "<CUR>")
    s = re.sub(r"\s+", " ", s).strip()
    return s


In [ ]:
check_df = df.copy()
check_df = check_df.dropna(subset=["Category", "Message"]).copy()
check_df["Category"] = check_df["Category"].astype(str).str.strip().str.lower()
check_df = check_df[check_df["Category"].isin(["ham", "spam"])]
check_df["Message"] = check_df["Message"].apply(clean_message)

conflict_df = (
    check_df.groupby("Message")["Category"]
    .nunique()
    .reset_index(name="label_count")
)
conflict_df = conflict_df[conflict_df["label_count"] > 1]

print(f"Conflicting messages: {len(conflict_df)}")
conflict_df.head()


In [ ]:
df["Category"] = df["Category"].astype(str).str.strip().str.lower()
df = df[df["Category"].isin(["ham", "spam"])]
df["Category"].value_counts()


In [ ]:
df = df.dropna(subset=["Category", "Message"]).copy()
df["Category"] = df["Category"].astype(str).str.strip().str.lower()
df = df[df["Category"].isin(["ham", "spam"])]
df["Message"] = df["Message"].apply(clean_message)
df = df[df["Message"] != ""]
conflicting_messages = set(conflict_df["Message"])
df = df[~df["Message"].isin(conflicting_messages)]
df = df.drop_duplicates(subset=["Message"], keep="first").reset_index(drop=True)

print(df.shape)
df.head()


In [ ]:
out_path = BASE_DIR / "Dataset_SMS_clean.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved: {out_path}")
